# Member 4 - Logistic Regression
## ML Assignment: FIFA Player Position Prediction

- **Algorithm:** Logistic Regression (Multi-class)
- **Dataset:** FIFA 22 Complete Player Dataset
- **Source:** Kaggle - FIFA 22 Players
- **Task:** Predict player position - ATTACKER, MIDFIELDER, or DEFENDER

This is a standalone version. No external files needed. Data is downloaded automatically.


## 1. Introduction - What is Logistic Regression?

Logistic Regression is a classification algorithm. Even though it has "regression" in the name, it is used to **predict categories**, not numbers.

It works by:
1. Taking the input features and calculating a score (called `z`)
2. Passing that score through a **softmax function** to get probabilities for each class
3. Picking the class with the **highest probability**

**Why use it for FIFA position prediction?**

- It is easy to understand - the model shows which stats matter most
- It gives a confidence score for each position
- It trains very fast, even on thousands of players
- It is a good starting point to compare with other algorithms
- One limitation: it can only draw a straight-line boundary between classes


## 2. Import Librarie


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, time
warnings.filterwarnings('ignore')

# Machine learning tools
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold, learning_curve
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, f1_score, precision_score, recall_score
)
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

# Plot settings
plt.rcParams["figure.dpi"] = 110
sns.set_theme(style="whitegrid", palette="muted")

print("All libraries imported successfully!")

## 3. Load and Prepare the Dataset

The dataset comes from Kaggle and contains stats for over 18,000 real football players from FIFA 22.

**Features used to train the model:**

- `age` - player age
- `height_cm` - height in centimetres
- `weight_kg` - weight in kilograms
- `overall` - skill rating (0–100)
- `potential` - potential rating (0–100)
- `pace` - speed
- `shooting` - shooting accuracy
- `passing` - passing accuracy
- `dribbling` - ball control
- `defending` - defensive skill
- `physic` - physical strength

**Target label:** ATTACKER, MIDFIELDER, or DEFENDER


In [ ]:
# Load the shared data files — same files used by all 4 members
X_train     = np.load("X_train.npy")
X_test      = np.load("X_test.npy")
y_train     = np.load("y_train.npy")
y_test      = np.load("y_test.npy")
class_names = np.load("class_names.npy", allow_pickle=True)

FEATURES = ["age", "height_cm", "weight_kg", "overall", "potential",
            "pace", "shooting", "passing", "dribbling", "defending", "physic"]

print("Shared data loaded!")
print(f"Training samples : {len(X_train):,}")
print(f"Testing  samples : {len(X_test):,}")
print(f"Features         : {X_train.shape[1]}")
print(f"Classes          : {list(class_names)}")
print()

# Show class distribution
unique, counts = np.unique(y_train, return_counts=True)
print("Training class distribution:")
for idx, cnt in zip(unique, counts):
    print(f"  {class_names[idx]:<12}: {cnt:,} samples ({cnt/len(y_train)*100:.1f}%)")

## 4. Train the Logistic Regression Model


In [ ]:
# Create the Logistic Regression model
# solver='lbfgs'  : works well for multi-class problems
# C=1.0           : regularisation strength (higher = less regularisation)
# max_iter=1000   : maximum number of steps to find the best weights
# random_state=42 : makes results reproducible
lr_model = LogisticRegression(
    solver='lbfgs',
    C=1.0,
    max_iter=1000,
    random_state=42
)

print('Training Logistic Regression...')
start = time.time()
lr_model.fit(X_train, y_train)
elapsed = time.time() - start

print(f'Training complete!')
print(f'   Time taken   : {elapsed:.3f} seconds')
print(f'   Iterations   : {lr_model.n_iter_[0]}')
print(f'   Converged    : {lr_model.n_iter_[0] < lr_model.max_iter}')

## 5. Evaluate the Model

### 5.1 Accuracy and Classification Report


In [ ]:
# Make predictions on the test set
y_pred   = lr_model.predict(X_test)
y_proba  = lr_model.predict_proba(X_test)

# Calculate accuracy on both training and test sets
test_acc  = accuracy_score(y_test,  y_pred)
train_acc = accuracy_score(y_train, lr_model.predict(X_train))

# AUC-ROC: measures how well the model separates the three classes
auc_score = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")

print("=" * 55)
print("    LOGISTIC REGRESSION — EVALUATION RESULTS")
print("=" * 55)
print(f"  Train Accuracy  : {train_acc * 100:.2f}%")
print(f"  Test  Accuracy  : {test_acc  * 100:.2f}%")
print(f"  AUC-ROC (macro) : {auc_score:.4f}")
gap = (train_acc - test_acc) * 100
print(f"  Overfitting Gap : {gap:.2f}%", end="  ")
print("Good generalisation" if gap < 3 else "Consider lower C")
print()
print("Per-Class Report:")
print(classification_report(y_test, y_pred, target_names=class_names, digits=4))

### 5.2 Precision, Recall and F1-Score

In [ ]:
precision_macro    = precision_score(y_test, y_pred, average="macro",    zero_division=0)
precision_weighted = precision_score(y_test, y_pred, average="weighted", zero_division=0)
recall_macro       = recall_score   (y_test, y_pred, average="macro",    zero_division=0)
recall_weighted    = recall_score   (y_test, y_pred, average="weighted", zero_division=0)
f1_macro           = f1_score       (y_test, y_pred, average="macro",    zero_division=0)
f1_weighted        = f1_score       (y_test, y_pred, average="weighted", zero_division=0)

print("=== PRECISION, RECALL AND F1-SCORE ===")
print(f"  Precision  (macro)    : {precision_macro:.4f}")
print(f"  Precision  (weighted) : {precision_weighted:.4f}")
print(f"  Recall     (macro)    : {recall_macro:.4f}")
print(f"  Recall     (weighted) : {recall_weighted:.4f}")
print(f"  F1-Score   (macro)    : {f1_macro:.4f}")
print(f"  F1-Score   (weighted) : {f1_weighted:.4f}")
print()

# Per-class breakdown
precision_per = precision_score(y_test, y_pred, average=None, zero_division=0)
recall_per    = recall_score   (y_test, y_pred, average=None, zero_division=0)
f1_per        = f1_score       (y_test, y_pred, average=None, zero_division=0)

print(f'{"Class":<20} {"Precision":>10} {"Recall":>10} {"F1-Score":>10}')
print("-" * 52)
for i, cls in enumerate(class_names):
    print(f"{cls:<20} {precision_per[i]:>10.4f} {recall_per[i]:>10.4f} {f1_per[i]:>10.4f}")

### 5.3 Confusion Matrix

In [ ]:
# Build the confusion matrix
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=class_names, yticklabels=class_names,
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Confusion Matrix (Counts)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Right: percentage per row (shows recall per class)
sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='YlOrRd',
            xticklabels=class_names, yticklabels=class_names,
            ax=axes[1], linewidths=0.5)
axes[1].set_title('Confusion Matrix (Normalised %)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

plt.suptitle('Logistic Regression — Confusion Matrices', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('member4_confusion_matrix.png', bbox_inches='tight')
plt.show()

print('\nPer-Class Recall:')
for i, cls in enumerate(class_names):
    recall = cm[i,i] / cm[i].sum() * 100
    print(f'  {cls:<12}: {cm[i,i]}/{cm[i].sum()} correct  ({recall:.1f}% recall)')

### 5.4 ROC Curve (One-vs-Rest per Class)

In [ ]:
classes    = np.unique(y_train)
y_test_bin = label_binarize(y_test, classes=classes)

roc_auc_macro    = roc_auc_score(y_test_bin, y_proba, multi_class="ovr", average="macro")
roc_auc_weighted = roc_auc_score(y_test_bin, y_proba, multi_class="ovr", average="weighted")

print("=== ROC-AUC SCORE (One-vs-Rest) ===")
print(f"  ROC-AUC (macro)    : {roc_auc_macro:.4f}")
print(f"  ROC-AUC (weighted) : {roc_auc_weighted:.4f}")

plt.figure(figsize=(9, 6))
colors = ["#E74C3C", "#2980B9", "#27AE60"]

for i, (cls_idx, color) in enumerate(zip(classes, colors)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    roc_auc_cls  = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2,
             label=f"{class_names[i]} (AUC = {roc_auc_cls:.2f})")

plt.plot([0, 1], [0, 1], "k--", lw=1, label="Random Classifier")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Logistic Regression — ROC Curve (One-vs-Rest)", fontweight="bold")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("member4_roc_curve.png", bbox_inches="tight")
plt.show()
print("ROC curve saved!")

### 5.5 TP / FP / TN / FN Breakdown

In [ ]:
print(f"{chr(39)}Class{chr(39):<20} {chr(39)}TP{chr(39):>6} {chr(39)}FP{chr(39):>6} {chr(39)}TN{chr(39):>6} {chr(39)}FN{chr(39):>6}")
print("-" * 44)
for i, cls in enumerate(class_names):
    TP = cm[i, i]
    FP = cm[:, i].sum() - TP
    FN = cm[i, :].sum() - TP
    TN = cm.sum() - TP - FP - FN
    print(f"{cls:<20} {TP:>6} {FP:>6} {TN:>6} {FN:>6}")

### 5.6 Cross-Validation (5-Fold)


In [ ]:
# Combine train and test sets for full cross-validation
X_all = np.vstack([X_train, X_test])
y_all = np.concatenate([y_train, y_test])

# 5-fold stratified cross-validation: tests the model on 5 different splits
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_model = LogisticRegression(max_iter=1000, solver='lbfgs', C=1.0, random_state=42)
cv_scores = cross_val_score(cv_model, X_all, y_all, cv=skf, scoring='accuracy')

print('5-Fold Stratified Cross-Validation:')
for i, s in enumerate(cv_scores, 1):
    bar = '█' * int(s * 40)
    print(f'  Fold {i}: {s*100:.2f}%  {bar}')
print(f'\n  Mean : {cv_scores.mean()*100:.2f}%')
print(f'  Std  : +/-{cv_scores.std()*100:.2f}%')

# Plot the accuracy of each fold
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, 6), cv_scores * 100, color='#2980B9', alpha=0.8, edgecolor='white')
ax.axhline(y=cv_scores.mean()*100, color='red', linestyle='--',
           label=f'Mean = {cv_scores.mean()*100:.2f}%')
ax.set_xlabel('Fold')
ax.set_ylabel('Accuracy (%)')
ax.set_title('5-Fold Cross-Validation Accuracy', fontsize=13, fontweight='bold')
ax.set_xticks(range(1, 6))
ax.set_ylim([0, 105])
ax.legend()
plt.tight_layout()
plt.savefig('member4_cv_scores.png', bbox_inches='tight')
plt.show()

## 6. Feature Importance (Coefficients)

Logistic Regression shows which player stats had the most influence on predicting each position.


In [ ]:
# Plot the learned coefficients for each position class
# Positive coefficient = that feature increases chance of this class
# Negative coefficient = that feature decreases chance of this class
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, (ax, cls) in enumerate(zip(axes, class_names)):
    coefs = lr_model.coef_[i]
    idx   = np.argsort(coefs)
    sorted_features = [FEATURES[j] for j in idx]
    sorted_coefs    = coefs[idx]
    colors = ['#E74C3C' if c > 0 else '#3498DB' for c in sorted_coefs]

    bars = ax.barh(sorted_features, sorted_coefs, color=colors, edgecolor='white', height=0.65)
    ax.axvline(x=0, color='black', linewidth=1.2, linestyle='--')
    ax.set_title(f'{cls}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Coefficient')
    for bar, val in zip(bars, sorted_coefs):
        x = val + 0.01 if val >= 0 else val - 0.01
        ax.text(x, bar.get_y() + bar.get_height()/2,
                f'{val:.2f}', va='center', fontsize=7,
                ha='left' if val >= 0 else 'right')

from matplotlib.patches import Patch
fig.legend(handles=[
    Patch(facecolor='#E74C3C', label='Increases probability'),
    Patch(facecolor='#3498DB', label='Decreases probability')
], loc='lower center', ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.05))

plt.suptitle('Feature Coefficients per Position Class', fontsize=14, y=1.02, fontweight='bold')
plt.tight_layout()
plt.savefig('member4_coefficients.png', bbox_inches='tight')
plt.show()

print('\nTop predictors per class:')
for i, cls in enumerate(class_names):
    coefs = lr_model.coef_[i]
    print(f'  {cls:<12}: strongest +ve = {FEATURES[np.argmax(coefs)]:<12} | strongest -ve = {FEATURES[np.argmin(coefs)]}')

## 7. Sample Prediction Probabilities


In [ ]:
# Show prediction probabilities for the first 10 test players
n_show = 10
proba  = lr_model.predict_proba(X_test[:n_show])

print(f'Prediction probabilities for first {n_show} test players:\n')
header = f'{"Player":<9}'
for cn in class_names:
    header += f'{cn:<14}'
header += f'{"Predicted":<14}{"Actual":<14}OK?'
print(header)
print('-' * 80)

correct = 0
for i in range(n_show):
    predicted = class_names[np.argmax(proba[i])]
    actual    = class_names[y_test[i]]
    mark      = 'correct' if predicted == actual else 'wrong'
    if predicted == actual: correct += 1
    row = f'Player {i+1:<3}'
    for p in proba[i]:
        row += f'{p:.3f}         '
    row += f'{predicted:<14}{actual:<14}{mark}'
    print(row)

print(f'\n  Sample accuracy: {correct}/{n_show} ({correct/n_show*100:.0f}%)')

## 8. Learning Curve


In [ ]:
# Learning curve: shows how accuracy changes as we use more training data
lc_model = LogisticRegression(max_iter=1000, solver='lbfgs', C=1.0, random_state=42)

train_sizes, train_scores, val_scores = learning_curve(
    lc_model, X_train, y_train, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy'
)

# Calculate mean and standard deviation across folds
tm, ts = train_scores.mean(axis=1), train_scores.std(axis=1)
vm, vs = val_scores.mean(axis=1),   val_scores.std(axis=1)

plt.figure(figsize=(9, 5))
plt.plot(train_sizes, tm, 'o-', color='#E74C3C', label='Training Accuracy')
plt.fill_between(train_sizes, tm-ts, tm+ts, alpha=0.15, color='#E74C3C')
plt.plot(train_sizes, vm, 's-', color='#2980B9', label='Validation Accuracy')
plt.fill_between(train_sizes, vm-vs, vm+vs, alpha=0.15, color='#2980B9')
plt.xlabel('Training Samples', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Learning Curve — Logistic Regression', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('member4_learning_curve.png', bbox_inches='tight')
plt.show()

# Check for overfitting: small gap = good balance
gap = tm[-1] - vm[-1]
print(f'\n  Final Train Accuracy      : {tm[-1]*100:.2f}%')
print(f'  Final Validation Accuracy : {vm[-1]*100:.2f}%')
print(f'  Bias-Variance Gap         : {gap*100:.2f}%')
if gap < 0.03:   print('  Good balance — no significant overfitting')
elif gap < 0.08: print('  Slight overfitting — try lower C')
else:            print('  Overfitting detected — reduce C or features')

## 9. Effect of Regularisation (C Parameter)


In [ ]:
# Test different values of C to see how regularisation affects accuracy
# Small C = strong regularisation, Large C = weak regularisation
C_values = [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 100.0]
tr_accs, te_accs = [], []

for c in C_values:
    m = LogisticRegression(max_iter=1000, solver='lbfgs', C=c, random_state=42)
    m.fit(X_train, y_train)
    tr_accs.append(accuracy_score(y_train, m.predict(X_train)))
    te_accs.append(accuracy_score(y_test,  m.predict(X_test)))

plt.figure(figsize=(9, 5))
plt.semilogx(C_values, [a*100 for a in tr_accs], 'o-', color='#E74C3C', label='Train')
plt.semilogx(C_values, [a*100 for a in te_accs], 's-', color='#2980B9', label='Test')
plt.axvline(x=1.0, color='green', linestyle='--', linewidth=1.5, label='C=1.0 (chosen)')
plt.xlabel('C value (log scale)', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.title('Effect of Regularisation Parameter C', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('member4_regularisation.png', bbox_inches='tight')
plt.show()

best_idx = int(np.argmax(te_accs))
print(f'  Best C         : {C_values[best_idx]}')
print(f'  Best Test Acc  : {te_accs[best_idx]*100:.2f}%')
print(f'  C=1.0 Test Acc : {te_accs[C_values.index(1.0)]*100:.2f}%')

## 10. Analysis and Discussion

### Strengths
- Trains very quickly (under 2 seconds)
- Easy to interpret — you can see which stats affect each position
- Returns a probability for each class, not just a label
- No overfitting — training and test accuracy are close

### Limitations
1. Can only separate classes with a straight line — it misses complex patterns like the interaction between pace and shooting
2. If one position has fewer players, it may not predict that class as well
3. Only 11 features are used — adding more could improve results
4. Some players can play multiple positions, which makes classification harder

### How to Improve Accuracy
- Tune the C value using grid search
- Add new features like pace × shooting
- Use `class_weight='balanced'` to handle unequal class sizes
- Try polynomial features to capture non-linear patterns
- Combine with other algorithms

### Comparison with Other Algorithms
Logistic Regression is a good baseline. Non-linear models like Random Forest, SVM, and KNN can find more complex patterns. But Logistic Regression is faster and easier to explain.

### Ideas for Future Work
- Predict more specific positions (e.g., LW, CM, CB)
- Try dimensionality reduction (PCA or LDA)
- Use SHAP values to explain individual predictions
- Include data from multiple FIFA editions


## 11. Final Summary


In [ ]:
print('=' * 60)
print('  MEMBER 4 - LOGISTIC REGRESSION - FINAL SUMMARY')
print('=' * 60)
print(f'  Algorithm       : Logistic Regression (Multinomial)')
print(f'  Solver          : L-BFGS')
print(f'  Regularisation  : L2  (C = 1.0)')
print(f'  Max iterations  : 1000')
print(f'  Training size   : {len(X_train):,} samples')
print(f'  Test size       : {len(X_test):,} samples')
print(f'  Features        : {X_train.shape[1]}')
print(f'  Classes         : {list(class_names)}')
print()
print(f'  Train Accuracy  : {train_acc * 100:.2f}%')
print(f'  Test  Accuracy  : {test_acc  * 100:.2f}%')
print(f'  AUC-ROC (macro) : {auc_score:.4f}')
print(f'  CV Mean Acc     : {cv_scores.mean()*100:.2f}% +/- {cv_scores.std()*100:.2f}%')
print()
print('  Output files saved:')
for f in ['member4_class_distribution.png','member4_confusion_matrix.png',
          'member4_coefficients.png','member4_cv_scores.png',
          'member4_learning_curve.png','member4_regularisation.png']:
    print(f'    {f}')
print('=' * 60)